In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver
import os

In [2]:
load_dotenv()
api_key = os.getenv('GOOGLE_API_KEY')

model = ChatGoogleGenerativeAI(
    model = 'gemini-3.8-flash',
    api_key = api_key
)

In [3]:
class JokeState(TypedDict):
    topic:str
    joke:str
    explanation:str

In [4]:
def generate_joke(state:JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = model.invoke(prompt).content

    return {'joke': response}

In [5]:
def generate_explaination(state: JokeState):

    prompt = f'write an explaination for the joke = {state['joke']}'

    response = model.invoke(prompt).content

    return { "explaination": response}

In [6]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explaination', generate_explaination)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explaination')
graph.add_edge('generate_explaination', END)

checkpointer = InMemorySaver()

workflow = graph.compile(checkpointer = checkpointer)

In [ ]:
config1 = {'configurable': {'thread_id': '1'}}
workflow.invoke({'topic': 'pizza'}, config = config1)

In [ ]:
workflow.get_state(config1)

StateSnapshot(values={'topic': 'pizza', 'joke': [{'type': 'text', 'text': 'I was going to tell you a joke about pizza... \n\nNever mind, it’s way too cheesy. And honestly, the delivery is terrible.', 'extras': {'signature': 'EooNCocNAWkUfRPhYtMFIDvhFXtHVEFU9jf5VaQapbcx8hok/E4yRS37CJx7M3bOGs/GtpAgnMQfGLduMOQtspnso7D02SMa7lBIoz4gEDGaLonXHGpe0pJaixMeOzg7YKAihVi/knyvRUL+to3fK+HnVUfzoC3YcDU1qko5emhVOSnW76bHgPBtl0rDKStzfkDaze7coJ0QUWkepUWh+CqlTSuvjGPsLQTkyXiwq7KzRECKWaAFTaMPFNfApQSOyxFvTMi7+64D8C+5JgstticcsTW4tjb7bOunHajHAvKDC0mCr8MXvZABHnmKvvqKMmdCpCjER5H/EI/MICQdzbrM5ohce09gQgwUsqFfwNXYchz1tCLvZM3fYfY+mEUemcRYILoyVuhcckPkvZfH8JbEzIzZLFt1y5bJQTuhF7OGO8f2y6+S1kEwTk5PNAIwerpn2KZnwAqwdO4UPLQAUFjFDoesD6LS/QQvUpL3/iqf84wXs7ZY3DCdQPeOgqGmF++rhBLi2GgAjKhQH8Ag3OUC87EKccEgnP9tNZCXpaRWePhB94jaBX8lRL/wGqcGM7me8uhF1fozsE1PfZe3YyokkgAMHJYjabpevgYQNQop8o7Z4i9RHHqwYQQvS8T1rosPfJxR+tOz98W1KG6pUcznoHB7kWg+xZn7Pwktn8Ud7YwaBjgvPSuJ/RWT5pkAubo6dZ9P+iu19YpiwIzik9Fph08LuQdlPDDv4pqZ0m4jkSyh39QF95qiSKGvX+Ha1K84Bq9

In [ ]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': [{'type': 'text', 'text': 'I was going to tell you a joke about pizza... \n\nNever mind, it’s way too cheesy. And honestly, the delivery is terrible.', 'extras': {'signature': 'EooNCocNAWkUfRPhYtMFIDvhFXtHVEFU9jf5VaQapbcx8hok/E4yRS37CJx7M3bOGs/GtpAgnMQfGLduMOQtspnso7D02SMa7lBIoz4gEDGaLonXHGpe0pJaixMeOzg7YKAihVi/knyvRUL+to3fK+HnVUfzoC3YcDU1qko5emhVOSnW76bHgPBtl0rDKStzfkDaze7coJ0QUWkepUWh+CqlTSuvjGPsLQTkyXiwq7KzRECKWaAFTaMPFNfApQSOyxFvTMi7+64D8C+5JgstticcsTW4tjb7bOunHajHAvKDC0mCr8MXvZABHnmKvvqKMmdCpCjER5H/EI/MICQdzbrM5ohce09gQgwUsqFfwNXYchz1tCLvZM3fYfY+mEUemcRYILoyVuhcckPkvZfH8JbEzIzZLFt1y5bJQTuhF7OGO8f2y6+S1kEwTk5PNAIwerpn2KZnwAqwdO4UPLQAUFjFDoesD6LS/QQvUpL3/iqf84wXs7ZY3DCdQPeOgqGmF++rhBLi2GgAjKhQH8Ag3OUC87EKccEgnP9tNZCXpaRWePhB94jaBX8lRL/wGqcGM7me8uhF1fozsE1PfZe3YyokkgAMHJYjabpevgYQNQop8o7Z4i9RHHqwYQQvS8T1rosPfJxR+tOz98W1KG6pUcznoHB7kWg+xZn7Pwktn8Ud7YwaBjgvPSuJ/RWT5pkAubo6dZ9P+iu19YpiwIzik9Fph08LuQdlPDDv4pqZ0m4jkSyh39QF95qiSKGvX+Ha1K84Bq